In [0]:
from delta import configure_spark_with_delta_pip, DeltaTable
from pyspark.sql import SparkSession

In [0]:
builder = (SparkSession.builder
           .appName("merge-delta-table")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [0]:
%load_ext sparksql_magic
%config SparkSql.limit=20

In [0]:
%%sparksql 
CREATE OR REPLACE TABLE default.movie_and_show_titles ( 
    show_id STRING, 
    type STRING, 
    title STRING, 
    director STRING, 
    cast STRING, 
    country STRING, 
    date_added STRING, 
    release_year STRING, 
    rating STRING, 
    duration STRING, 
    listed_in STRING, 
    description STRING  
) USING DELTA LOCATION '/opt/workspace/data/delta_lake/movie_and_show_titles'; 

In [0]:
# For PySpark:
deltaTable_titles = DeltaTable.forPath(spark, "/opt/workspace/data/delta_lake/movie_and_show_titles")

In [0]:
deltaTable_titles.toDF().show(5)

In [0]:
# For PySpark:
df_netflix = spark.read.format("delta").load("/opt/workspace/data/delta_lake/netflix_titles")
df_netflix_deduped = df_netflix.dropDuplicates(["type", "title", "director", "date_added"])

In [0]:
(deltaTable_titles.alias('movie_and_show_titles')
 .merge(df_netflix_deduped.alias('updates')
        ,"""lower(movie_and_show_titles.type) = lower(updates.type)
          AND lower(movie_and_show_titles.title) = lower(updates.title)
          AND lower(movie_and_show_titles.director) = lower(updates.director)
          AND movie_and_show_titles.date_added = updates.date_added""")
 .whenMatchedUpdate(set ={
    "show_id": "updates.show_id",
     "type": "updates.type",
     "title" : "updates.title",
     "director" : "updates.director",
     "cast" : "updates.cast",
     "country" : "updates.country",
     "date_added" : "updates.date_added",
     "release_year" : "updates.release_year",
     "rating" : "updates.rating",
     "duration" : "updates.duration",
     "listed_in" : "updates.listed_in",
     "description" : "updates.description"})
 .whenNotMatchedInsert(values = {
    "show_id": "updates.show_id",
     "type": "updates.type",
     "title" : "updates.title",
     "director" : "updates.director",
     "cast" : "updates.cast",
     "country" : "updates.country",
     "date_added" : "updates.date_added",
     "release_year" : "updates.release_year",
     "rating" : "updates.rating",
     "duration" : "updates.duration",
     "listed_in" : "updates.listed_in",
     "description" : "updates.description"})
  .execute())

In [0]:
%%sparksql
DESCRIBE HISTORY "/opt/workspace/data/delta_lake/movie_and_show_titles"

In [0]:
# Read CSV file into a DataFrame
df_titles = (spark.read
      .format("csv")
      .option("header", "true")
      .load("../data/titles.csv"))

In [0]:
df_titles_deduped = df_titles.dropDuplicates(["type", "title"])

In [0]:
df_titles_deduped.printSchema()

In [0]:
df_titles_deduped.createOrReplaceTempView("titles_deduped")

In [0]:
(deltaTable_titles.alias('movie_and_show_titles')
 .merge(df_titles_deduped.alias('updates')
        ,"""lower(movie_and_show_titles.type) = lower(updates.type)
          AND lower(movie_and_show_titles.title) = lower(updates.title)
          AND movie_and_show_titles.release_year = updates.release_year""")
 .whenMatchedUpdate(set ={
     "show_id" : "updates.id",
     "type" : "updates.type",
     "title" : "updates.title",
     "country" : "updates.production_countries",
     "release_year" : "updates.release_year",
     "rating" : "updates.age_certification",
     "duration" : "updates.runtime",
     "listed_in" : "updates.genres",
     "description" : "updates.description"})
 .whenNotMatchedInsert(values = {
     "show_id" : "updates.id",
     "type" : "updates.type",
     "title" : "updates.title",
     "country" : "updates.production_countries",
     "release_year" : "updates.release_year",
     "rating" : "updates.age_certification",
     "duration" : "updates.runtime",
     "listed_in" : "updates.genres",
     "description" : "updates.description"})
  .execute())

In [0]:
%%sparksql
MERGE INTO default.movie_and_show_titles
USING titles_deduped
ON lower(default.movie_and_show_titles.type) = lower(titles_deduped.type) 
    AND lower(default.movie_and_show_titles.title) = lower(titles_deduped.title) 
    AND default.movie_and_show_titles.release_year = titles_deduped.release_year
WHEN MATCHED THEN
  UPDATE SET
    show_id = titles_deduped.id,
    type = titles_deduped.type,
    title = titles_deduped.title,
    country = titles_deduped.production_countries,
    release_year = titles_deduped.release_year,
    rating = titles_deduped.age_certification,
    duration = titles_deduped.runtime,
    listed_in = titles_deduped.genres,
    description = titles_deduped.description
WHEN NOT MATCHED
  THEN INSERT (
    show_id,
    type,
    title,
    country,
    release_year,
    rating,
    duration,
    listed_in,
    description
  )
  VALUES (
    titles_deduped.id,
    titles_deduped.type,
    titles_deduped.title,
    titles_deduped.production_countries,
    titles_deduped.release_year,
    titles_deduped.age_certification,
    titles_deduped.runtime,
    titles_deduped.genres,
    titles_deduped.description
  )

In [0]:
%%sparksql
DESCRIBE HISTORY "/opt/workspace/data/delta_lake/movie_and_show_titles"

In [0]:
spark.stop()